## General stats on the dataset and dataset structure

In [1]:
import numpy as np
import os, sys
import copy

# ===== IMPORTANT CONFIGURATION VARIABLES =====
input_dataset_type = "raw"
output_dataset_type = "preprocessed"
dataset_split = ['training', 'validation', 'test']

# Function to find project root
def find_project_root(name='Bambino', start=None):
    if start is None:
        start = os.getcwd()
    parent_dir = start
    while True:
        if os.path.basename(parent_dir) == name:
            return parent_dir
        new_parent = os.path.dirname(parent_dir)
        if new_parent == parent_dir:
            return None
        parent_dir = new_parent

PROJECT_ROOT = find_project_root('Bambino')
print('Detected project root:', PROJECT_ROOT)

def concat_datasets_new(datasets):
    valid = [d for d in datasets if d is not None]
    if not valid:
        return None

    # copia "strutturale" del primo dataset (senza side effects)
    combined = copy.copy(valid[0])
    combined.instances = []

    for ds in valid:
        combined.instances.extend(ds.instances)

    return combined


# Provo a importare i moduli del progetto una sola volta (se PROJECT_ROOT esiste)
utils = None
BoaOpenFaceDataset = None
if PROJECT_ROOT:
    try:
        sys.path.append(PROJECT_ROOT)
        from config import settings as utils
        from DataUtils.BoaOpenFaceDataset import BoaOpenFaceDataset
        print('Imported project modules successfully.')
    except Exception as e:
        print('Could not import project modules (this is OK). Error:', e)
        utils = None
        BoaOpenFaceDataset = None
else:
    print('Project root non trovato: salto import dei moduli di progetto.')

# Dizionario per conservare i dataset per split
datasets = {split: None for split in dataset_split}

# Carico ogni split (se possibile) e salvo il relativo dataset nel dizionario
for split in dataset_split:
    print(f'Processing split: {split}')
    if utils is not None and BoaOpenFaceDataset is not None:
        try:
            in_path = utils.get_dataset_path(input_dataset_type,
                                            getattr(utils, f'{split}_filename'))
            print('Attempting to load dataset from:', in_path)
            ds = BoaOpenFaceDataset.load_dataset(in_path, modalities=None)
            print(f'Dataset "{split}" loaded: type={type(ds)}, instances={len(ds.instances)}')
            datasets[split] = ds
        except Exception as e:
            print(f'Could not load dataset for split "{split}" (this is OK). Error:', e)
            datasets[split] = None
    else:
        print(f'Skipping load for split "{split}" because project modules are not available.')
        datasets[split] = None

# Dataset separati per accesso diretto (None se mancanti)
training_dataset = datasets.get('training')
validation_dataset = datasets.get('validation')
test_dataset = datasets.get('test')

# Concatenazione complessiva (solo dei dataset non-None)
combined_dataset = concat_datasets_new([
    training_dataset,
    validation_dataset,
    test_dataset
])

# Stampa riepilogo
print('--- Summary ---')
for name, ds in datasets.items():
    if ds is None:
        print(f'{name}: MISSING')
    else:
        print(f'{name}: loaded, instances = {len(ds.instances)}')
if combined_dataset is None:
    print('combined_dataset: None (no split loaded)')
else:
    print('combined_dataset: created, total instances =', len(combined_dataset.instances))

# ora hai:
# - training_dataset
# - validation_dataset
# - test_dataset
# - combined_dataset (concat dei tre, o None se nessuno era disponibile)


Detected project root: /home/phd2/Scrivania/CorsoRepo/Bambino
Imported project modules successfully.
Processing split: training
Attempting to load dataset from: /home/phd2/Scrivania/CorsoRepo/Bambino/data/raw/training_set.pt
Dataset "training" loaded: type=<class 'DataUtils.BoaOpenFaceDataset.BoaOpenFaceDataset'>, instances=619
Processing split: validation
Attempting to load dataset from: /home/phd2/Scrivania/CorsoRepo/Bambino/data/raw/validation_set.pt
Dataset "validation" loaded: type=<class 'DataUtils.BoaOpenFaceDataset.BoaOpenFaceDataset'>, instances=140
Processing split: test
Attempting to load dataset from: /home/phd2/Scrivania/CorsoRepo/Bambino/data/raw/test_set.pt
Dataset "test" loaded: type=<class 'DataUtils.BoaOpenFaceDataset.BoaOpenFaceDataset'>, instances=137
--- Summary ---
training: loaded, instances = 619
validation: loaded, instances = 140
test: loaded, instances = 137
combined_dataset: created, total instances = 896


In [2]:
# Inspect dataset structure
if combined_dataset is not None:
    print('\nInspecting dataset structure:')
    print('AGE_GROUPS =', combined_dataset.AGE_GROUPS)
    print('AGE_GROUPS_BOA =', combined_dataset.AGE_GROUPS_BOA)
    print('FRAME_RATE =', combined_dataset.FRAME_RATE)
    print('MIN_SEQUENCE_LENGTH =', combined_dataset.MIN_SEQUENCE_LENGTH)
    print('SEX_GROUPS =', combined_dataset.SEX_GROUPS)
    print('SPEAKER_GROUPS =', combined_dataset.SPEAKER_GROUPS)
    print('TRIAL_ID_GROUPS =', combined_dataset.TRIAL_ID_GROUPS)
    print('TRIAL_TYPES =', combined_dataset.TRIAL_TYPES)
    print('\nInspecting first instance:')

    for inst in combined_dataset.instances[:1]:
        print('Age:', inst.age)
        print('Sex:', inst.sex)
        print('gaze_info:', inst.gaze_info.shape)
        print('head_info:', inst.head_info.shape)
        print('face_info:', inst.face_info.shape)
        print('audio_type:', inst.audio)
        print('pt_id:', inst.pt_id)
        print('Speaker:', inst.speaker)
        print('Trial_id:', inst.trial_id)
        print('Trial_type:', inst.trial_type)

else:
    print('Dataset not loaded; cannot inspect structure.')


Inspecting dataset structure:
AGE_GROUPS = ['7-11', '12-18', '19-24']
AGE_GROUPS_BOA = ['[3-5.5)', '[5.5-7]']
FRAME_RATE = 25
MIN_SEQUENCE_LENGTH = 300
SEX_GROUPS = ['Female', 'Male']
SPEAKER_GROUPS = ['Left', 'Right']
TRIAL_ID_GROUPS = ['0-30th percentiles', '30-70th percentiles', '70-100th percentiles']
TRIAL_TYPES = ['control', 'stimulus']

Inspecting first instance:
Age: 5.65
Sex: 1
gaze_info: (250, 8)
head_info: (250, 13)
face_info: (250, 17)
audio_type: other personalised
pt_id: bam1_001
Speaker: 0
Trial_id: 1
Trial_type: 1


In [4]:
import numpy as np
from collections import Counter, defaultdict

def analyze_dataset(name, ds):
    print(f'\n===== Stats for: {name} =====')
    if ds is None:
        print('> Dataset is None (not available).')
        return {}

    instances = getattr(ds, 'instances', [])
    n_instances = len(instances)
    print(f'Total number of instances: {n_instances}')

    # --- Ages ---
    ages = []
    for inst in instances:
        a = getattr(inst, 'age', None)
        if a is not None:
            try:
                ages.append(float(a))
            except Exception:
                # ignore non-numeric ages
                pass
    n_ages = len(ages)
    n_missing_age = n_instances - n_ages
    if n_ages > 0:
        arr = np.array(ages, dtype=float)
        mean_age = arr.mean()
        std_age = arr.std(ddof=0)
        p2_5, p50, p97_5 = np.percentile(arr, [2.5, 50, 97.5])
        min_age = arr.min()
        max_age = arr.max()
        print(f'Ages: count={n_ages}, missing={n_missing_age}')
        print(f'  mean={mean_age:.2f}, std={std_age:.2f}, min={min_age:.2f}, 2.5%={p2_5:.2f}, median={p50:.2f}, 97.5%={p97_5:.2f}, max={max_age:.2f}')
    else:
        print('No valid age values found.')

    # --- Sex (instance-level) ---
    sex_instances = [getattr(inst, 'sex', 'unknown') for inst in instances]
    sex_inst_counts = Counter(sex_instances)
    print('Instance-level sex counts:')
    for s, c in sex_inst_counts.items():
        cat_sex = "M" if s == 1 else "F" if s == 0 else "unknown"
        print(f'  {cat_sex}: {c}')

    # --- Unique subjects and sex per-subject ---
    pt_to_sex = {}
    for inst in instances:
        pid = getattr(inst, 'pt_id', None)
        if pid is None:
            continue
        if pid not in pt_to_sex:
            pt_to_sex[pid] = "M" if getattr(inst, 'sex', None) == 1 else "F" if getattr(inst, 'sex', None) == 0 else "unknown"
        else:
            # if current stored is unknown, prefer a known sex if found later
            cur = pt_to_sex[pid]
            new = "M" if getattr(inst, 'sex', None) == 1 else "F" if getattr(inst, 'sex', None) == 0 else "unknown"
            if cur == 'unknown' and new in ('M', 'F'):
                pt_to_sex[pid] = new
            # if conflicting labels (rare), we keep the first and could log if needed

    unique_pt_ids = set(pid for pid in pt_to_sex.keys())
    # also consider subjects with pt_id present but sex never available -> count them as 'unknown'
    # gather all pt_ids (even those whose sex is None everywhere)
    all_pt_ids = set(getattr(inst, 'pt_id', None) for inst in instances if getattr(inst, 'pt_id', None) is not None)
    # ensure pt_to_sex includes unknown for subjects seen but with no sex collected
    for pid in all_pt_ids:
        if pid not in pt_to_sex:
            pt_to_sex[pid] = 'unknown'

    n_unique = len(all_pt_ids)
    print(f'Number of unique subjects (pt_id): {n_unique}')

    # Count unique subjects per sex
    unique_counts_by_sex = Counter(pt_to_sex.values())
    print('Unique subjects per sex (by pt_id):')
    for s, c in unique_counts_by_sex.items():
        print(f'  {s}: {c}')

    # return a dict of metrics if you want to programmatically consume results
    metrics = {
        'n_instances': n_instances,
        'n_age_values': n_ages,
        'n_missing_age': n_missing_age,
        'age_mean': float(mean_age) if n_ages>0 else None,
        'age_std': float(std_age) if n_ages>0 else None,
        'age_min': float(min_age) if n_ages>0 else None,
        'age_p2.5': float(p2_5) if n_ages>0 else None,
        'age_median': float(p50) if n_ages>0 else None,
        'age_p97.5': float(p97_5) if n_ages>0 else None,
        'age_max': float(max_age) if n_ages>0 else None,
        'sex_instance_counts': dict(sex_inst_counts),
        'n_unique_subjects': n_unique,
        'unique_subjects_by_sex': dict(unique_counts_by_sex),
    }
    return metrics

# --- Run for all splits + combined ---
# Expect these variables from your previous code:
# training_dataset, validation_dataset, test_dataset, combined_dataset
datasets_to_check = {
    'training': training_dataset,
    'validation': validation_dataset,
    'test': test_dataset,
    'combined': combined_dataset
}

all_metrics = {}
for name, ds in datasets_to_check.items():
    metrics = analyze_dataset(name, ds)
    all_metrics[name] = metrics



===== Stats for: training =====
Total number of instances: 619
Ages: count=619, missing=0
  mean=5.49, std=1.10, min=3.45, 2.5%=3.65, median=5.59, 97.5%=7.49, max=7.49
Instance-level sex counts:
  M: 274
  F: 345
Number of unique subjects (pt_id): 32
Unique subjects per sex (by pt_id):
  M: 14
  F: 18

===== Stats for: validation =====
Total number of instances: 140
Ages: count=140, missing=0
  mean=4.68, std=0.93, min=3.68, 2.5%=3.68, median=4.73, 97.5%=6.44, max=6.44
Instance-level sex counts:
  F: 120
  M: 20
Number of unique subjects (pt_id): 7
Unique subjects per sex (by pt_id):
  F: 6
  M: 1

===== Stats for: test =====
Total number of instances: 137
Ages: count=137, missing=0
  mean=5.83, std=0.71, min=4.83, 2.5%=4.83, median=6.08, 97.5%=6.70, max=6.70
Instance-level sex counts:
  F: 97
  M: 40
Number of unique subjects (pt_id): 7
Unique subjects per sex (by pt_id):
  F: 5
  M: 2

===== Stats for: combined =====
Total number of instances: 896
Ages: count=896, missing=0
  mean=5

In [7]:
# count the labels in the combined dataset (se esiste) per avere un'idea della distribuzione complessiva
if combined_dataset is not None:
    label_counts = Counter(getattr(inst, 'trial_type', 'unknown') for inst in combined_dataset.instances)
    print('\nLabel distribution in combined dataset (trial_type):')
    for label, count in label_counts.items():
        print(f'  {label}: {count}')
else:
    print('Combined dataset not available; cannot count labels.')

# print the percentage of each label in the combined dataset
if combined_dataset is not None:
    total = sum(label_counts.values())
    print('\nLabel distribution in combined dataset (trial_type) - percentages:')
    for label, count in label_counts.items():
        percentage = (count / total * 100) if total > 0 else 0
        print(f'  {label}: {percentage:.2f}%')
else:    print('Combined dataset not available; cannot calculate label percentages.')



Label distribution in combined dataset (trial_type):
  1: 719
  0: 177

Label distribution in combined dataset (trial_type) - percentages:
  1: 80.25%
  0: 19.75%
